# Final pipeline: ball possession on BAR-ATH

**What it does.** Learns possession from **IDSSE** (7 Bundesliga matches, CC-BY, per-frame
`BallStatus` + `BallPossession` — the same rule as the challenge GT), made to look like the BAR-ATH
automatic tracking, then labels every BAR-ATH frame:

1. **Degrade** IDSSE like the ATD: ball missing in bursts (41 %), players dropping out, slowly
   drifting position error — calibrated on BAR-ATH's *unlabelled* tracking statistics.
2. **v1 features** (203 per frame): ball kinematics, nearest home/away player, team shape and speed,
   occlusion memory, touches (direction/speed change next to a player), restart layouts, pressure,
   0.2–10 s windows.
3. **LightGBM** (H / A / D), with **hand-over and short-possession sample weights**.
4. **Boundary-aware Viterbi**: a second LightGBM predicts P(hand-over); team switches are cheap
   where it is high and expensive elsewhere. Then **D before the second-half kickoff** (FAQ.md).

**How it was chosen.** Every option was scored on IDSSE leave-one-match-out with the organisers'
own metric code; the public 5 minutes were looked at once, at the end, as a check.

| step | IDSSE spellF1 | kept |
|---|---|---|
| v1 + LightGBM + Viterbi | 0.619 | ✅ |
| + hand-over / short-possession weights | 0.626 (5/7 matches) | ✅ |
| + boundary-aware decoder | **0.637** (7/7 matches) | ✅ |
| stacking; merged-touch labels; 2 s controlled labels; dead/team split; min-dead + touch snap; v2 sequence features | ≤ +0.003 or worse | ❌ |

`python -m solution.submission --team NAME` produces the same file as this notebook.

In [ ]:
import os, sys, json, time, subprocess
ROOT = os.path.abspath('..') if os.path.isfile(os.path.join('..', 'games.py')) else os.path.abspath('.')
for p in (ROOT, os.path.join(ROOT, 'starter'), os.path.join(ROOT, 'evaluation')):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(ROOT)

import numpy as np, pandas as pd
from solution import (atd, boundary, container as C, dataset_fit, degrade, evaluate_external as ev,
                      features, idsse, labels, submission)
pm = C.pm

TEAM = 'MyTeam'                       # the file is evaluation/results/<TEAM>.csv -- use your team name
QUIET = lambda s: None
HAVE_DATA = atd.have_data()
print('BAR-ATH data present:', HAVE_DATA, '| final configuration:', submission.FINAL)

## 1. External data

In [ ]:
ext = idsse.load_all()
lab = pd.DataFrame([dict(match=m['name'], **dataset_fit.label_stats(m['label'])) for m in ext])
display(lab[['match', 'graded_min', 'n_poss', 'poss_per_alive_min', 'mean_poss_s', 'share_D']].round(2))

## 2. BAR-ATH: load and check against DATA.md

In [ ]:
if HAVE_DATA:
    bar = atd.load()
    assert atd.checks(bar)['ok'], 'load does not match DATA.md'

## 3. Calibrate the degradation (unlabelled statistics only) and check for input shift

Measured on BAR-ATH: ball-missing rate, mean missing-burst length, visible players. No label is read.
The shift check prints verdicts and feature *names* only.

In [ ]:
if HAVE_DATA:
    params = degrade.calibrated_params(bar)
    os.makedirs('data/models', exist_ok=True)
    json.dump(params, open(submission.FINAL['degrade_json'], 'w'))
    print('calibration written to', submission.FINAL['degrade_json'])
    r = subprocess.run([sys.executable, '-m', 'solution.diagnose_shift'], capture_output=True, text=True,
                       encoding='utf-8', env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
    print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l))
PARAMS = json.load(open(submission.FINAL['degrade_json'])) if os.path.isfile(submission.FINAL['degrade_json']) else None

## 4. Expected performance: the final configuration on IDSSE (leave-one-match-out)

Weighted LightGBM + boundary decoder, penalties chosen on the other six matches for each held-out
one. Cached after the first run (~1 min); first run ~15 min.

In [ ]:
t0 = time.time()
res = boundary.evaluate(None, features.build_features_v1, labels.raw, PARAMS, train_cfg={'weights': True},
                        bootstrap=200, log=QUIET)
ev.report(res, 'final configuration on IDSSE')
print('organisers baseline on the same degraded data: spellF1 {:.4f}  [{:.0f}s]'.format(
      res['pooled_baseline']['spell_f1_mean'], time.time() - t0))

## 5. Build the submission (same code path as the CLI)

In [ ]:
if HAVE_DATA:
    t0 = time.time()
    codes, digest = submission.build(TEAM, log=QUIET)
    path = os.path.join('evaluation', 'results', TEAM + '.csv')
    print(submission.summary(codes))
    print(path, 'sha256', digest, '[{:.0f}s]'.format(time.time() - t0))

## 6. Checks — format, and the public window as an alignment check only

17 possessions: about ±0.18 of noise, and choosing on it is against RULES.md. It confirms the file is
aligned (frames, H = Barcelona), nothing more.

In [ ]:
if HAVE_DATA:
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
    for args in (['evaluation/validate_submission.py', path], ['evaluation/run_evaluation.py']):
        r = subprocess.run([sys.executable, *args], capture_output=True, text=True, encoding='utf-8', env=env)
        print(r.stdout, r.stderr)

## 7. Reproduce from the command line (RULES.md §3: the same file)

In [ ]:
cmd = [sys.executable, '-m', 'solution.submission', '--team', TEAM]
print('python -m solution.submission --team', TEAM)
RUN_REPRO = False
if HAVE_DATA and RUN_REPRO:
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    print('identical:', digest in out)